# Debris Playground — Experiment with Different Simulations

A notebook to run debris simulations and explore which values you can modify.
Uses analytic velocity fields (no GeoClaw required).

**Dependencies:** `shapely`, `clawpack`, `numpy`, `matplotlib`

## Imports

In [ ]:
%matplotlib inline
from math import sin, cos, pi
from pylab import *
import shapely.plotting
import debris_tracking

## Parameters you can modify

| Parameter | Description | Suggested values |
|-----------|-------------|------------------|
| **Simulation** | | |
| `t0` | Start time (s) | 0 |
| `dt` | Time step (s) | 0.05 |
| `nsteps` | Number of steps | 80 |
| **Domain** | | |
| `domain` | [x1, x2, y1, y2] — debris stays inside | [0, 1, 0, 1] |
| **Flow choice** | | |
| `flow_type` | 'swirling' or 'shear' or 'constant' | 'swirling' |
| **Debris shape** | | |
| `L` | Side lengths | [0.15, 0.1, 0.15, 0.1] (rectangle) |
| `phi` | Turning angles (rad) | [pi/2, pi/2, pi/2, pi/2] |
| **Debris physics** | | |
| `advect` | True = passive (follow flow), False = mass/drag | True |
| `rho` | Debris density (kg/m³), 0 = massless | 0 |
| `friction_static` | Static friction when grounded | 0.2 |
| `friction_kinetic` | Kinetic friction when grounded | 0.1 |
| **Initial positions** | | |
| `z0_list` | [x, y, theta] per object | See below |

## Configure simulation

In [ ]:
# ========== SIMULATION ==========
t0 = 0.
dt = 0.05
nsteps = 80

# ========== DOMAIN ==========
domain = [0., 1., 0., 1.]  # [x1, x2, y1, y2]

# ========== FLOW ==========
flow_type = 'swirling'  # 'swirling', 'shear', or 'constant'

if flow_type == 'swirling':
    # Divergence-free swirling flow (streamlines are closed)
    u = lambda x,y,t: -2*sin(pi*y)*cos(pi*y)*sin(pi*x)**2
    v = lambda x,y,t: 2*sin(pi*x)*cos(pi*x)*sin(pi*y)**2
elif flow_type == 'shear':
    # Shear: u increases with y, v=0
    u = lambda x,y,t: 0.5*y
    v = lambda x,y,t: 0.
elif flow_type == 'constant':
    u = lambda x,y,t: 0.3
    v = lambda x,y,t: 0.1

# Depth (m) — uniform; needed for grounding check
h = lambda x,y,t: 10.

print(f'Flow: {flow_type}, dt={dt}, nsteps={nsteps}')

## Create debris objects

In [ ]:
# ========== DEBRIS SHAPE ==========
# L = side lengths (ncorners-1 values), phi = turning angles at each corner
# Rectangle: L = [width, height, width, height], phi = [pi/2]*4
L = [0.15, 0.1, 0.15, 0.1]
phi = [pi/2, pi/2, pi/2, pi/2]

# ========== DEBRIS PHYSICS ==========
debris1 = debris_tracking.DebrisObject()
debris1.L = L
debris1.phi = phi
debris1.advect = True   # True = passive (massless, follow flow)
debris1.rho = 0.       # kg/m³; 0 = massless
debris1.friction_static = 0.2
debris1.friction_kinetic = 0.1
debris1.bottom_area = 0.15 * 0.1  # m²
debris1.face_width = 0.1
debris1.height = 0.1

# Second debris (optional — try different rho or advect=False)
debris2 = debris_tracking.DebrisObject()
debris2.L = L
debris2.phi = phi
debris2.advect = True
debris2.rho = 0.
debris2.friction_static = 0.2
debris2.friction_kinetic = 0.1
debris2.bottom_area = 0.15 * 0.1
debris2.face_width = 0.1
debris2.height = 0.1

debris_list = [debris1, debris2]

# ========== INITIAL POSITIONS ==========
# z = (x, y, theta) — position of corner 0 and orientation
z0_list = [
    (0.3, 0.5, 0.),           # debris1: center-ish, horizontal
    (0.6, 0.5, pi/4),        # debris2: different spot, rotated 45°
]

# Obstacles (rectangles) — set to [] for none
obst_list = [
    debris_tracking.make_rectangular_obstacle(0.45, 0.55, 0.45, 0.55),
]
# obst_list = []  # no obstacles

## Run simulation

In [ ]:
debris_path_list = debris_tracking.make_debris_path_list(
    debris_list, z0_list, obst_list, domain,
    t0, dt, nsteps, h, u, v, verbose=False
)
print('Done. Computed', len(debris_path_list), 'debris paths.')

## Plot results

In [ ]:
extent = domain
fig, ax = subplots(figsize=(7, 7))

# Velocity quiver (sample)
xq = linspace(extent[0], extent[1], 12)
yq = linspace(extent[2], extent[3], 12)
Xq, Yq = meshgrid(xq, yq)
Uq = u(Xq, Yq, 0)
Vq = v(Xq, Yq, 0)
smax = sqrt(Uq**2 + Vq**2).max()
if smax > 0:
    quiver(Xq, Yq, Uq, Vq, scale=1.1*smax/(extent[1]-extent[0])*12, scale_units='xy', color='lightgray', width=0.003)

# Obstacles
for obst in obst_list:
    shapely.plotting.plot_polygon(obst['polygon'], add_points=False, ax=ax, color='blue', alpha=0.5)

# Plot debris at start and end
colors = ['red', 'green']
for i, dp in enumerate(debris_path_list):
    debris = dp.debris
    # start
    z0 = dp.z_path[0]
    xc, yc = debris.get_corners(z0, close_poly=True)
    ax.plot(xc, yc, color=colors[i], alpha=0.5, linestyle='--', label=f'debris{i+1} start')
    # end
    zf = dp.z_path[-1]
    xc, yc = debris.get_corners(zf, close_poly=True)
    ax.plot(xc, yc, color=colors[i], linewidth=2, label=f'debris{i+1} end')

ax.set_xlim(extent[:2])
ax.set_ylim(extent[2:])
ax.set_aspect(1)
ax.legend(loc='upper right')
ax.set_title(f'Debris: {flow_type} flow, t=0 to t={t0 + nsteps*dt:.2f}s')
ax.set_xlabel('x')
ax.set_ylabel('y')
tight_layout()
show()

## Plot full trajectories (all time steps)

In [ ]:
fig, ax = subplots(figsize=(7, 7))
for obst in obst_list:
    shapely.plotting.plot_polygon(obst['polygon'], add_points=False, ax=ax, color='blue', alpha=0.5)

for i, dp in enumerate(debris_path_list):
    # plot centroid path
    x_cent = dp.x_path.mean(axis=1)
    y_cent = dp.y_path.mean(axis=1)
    ax.plot(x_cent, y_cent, color=colors[i], alpha=0.7)
    ax.scatter(x_cent[0], y_cent[0], color=colors[i], s=50, zorder=5)
    ax.scatter(x_cent[-1], y_cent[-1], color=colors[i], s=100, marker='s', zorder=5)

ax.set_xlim(extent[:2])
ax.set_ylim(extent[2:])
ax.set_aspect(1)
ax.set_title('Debris centroid trajectories')
ax.set_xlabel('x')
ax.set_ylabel('y')
show()

## Quick reference: all modifiable values

**Simulation:** `t0`, `dt`, `nsteps`  
**Domain:** `domain`  
**Flow:** `flow_type`, or define custom `u(x,y,t)`, `v(x,y,t)`, `h(x,y,t)`  
**Debris shape:** `L`, `phi`  
**Debris physics:** `advect`, `rho`, `friction_static`, `friction_kinetic`, `bottom_area`, `face_width`, `height`  
**Initial positions:** `z0_list`  
**Obstacles:** `obst_list`